### Data ingestion to vector db

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\RAKPOOJA\Downloads\AI\RAG\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
##Read all the pdfs inside the directory
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 1 PDF files to process

Processing: ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf


Ignoring wrong pointing object 13 0 (offset 0)


  ✓ Loaded 3 pages

Total documents loaded: 3


In [ ]:
all_pdf_documents

In [3]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [4]:
chunks=split_documents(all_pdf_documents)
chunks

Split 3 documents into 10 chunks

Example chunk:
Content: ASHIQ U 
DevOps Engineer 
+91-7907558767
ashiqummathoor@outlook.com
LinkedIn
OBJECTIVE
SUMMARY
Enthusiastic about tackling challenging roles and collaborating with diverse teams to drive 
organisation...
Metadata: {'producer': 'macOS Version 13.4 (Build 22F66) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20240324062134Z00'00'", 'title': 'MOHAMMED_ASHIQ_J2_V02', 'moddate': "D:20240324062134Z00'00'", 'source': '..\\data\\pdf\\ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 13.4 (Build 22F66) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20240324062134Z00'00'", 'title': 'MOHAMMED_ASHIQ_J2_V02', 'moddate': "D:20240324062134Z00'00'", 'source': '..\\data\\pdf\\ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'file_type': 'pdf'}, page_content='ASHIQ U \nDevOps Engineer \n+91-7907558767\nashiqummathoor@outlook.com\nLinkedIn\nOBJECTIVE\nSUMMARY\nEnthusiastic about tackling challenging roles and collaborating with diverse teams to drive \norganisational success. Possessing extensive experience across various cloud and on-premises \nenvironments, I am dedicated to implementing high availability and resilient architectures. With \na clear, logical mindset and a practical approach to problem-solving, I strive to see projects \nthrough to successful completion, contributing eﬀectively to team objectives.\n๏Over 4+ years of 

embedding And vectorStoreDB

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2",cache_folder:str="./models"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = SentenceTransformer(r"C:\models\all-MiniLM-L6-v2")
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(r"C:\models\all-MiniLM-L6-v2")

            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: C:\models\all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TypeError: Pooling.__init__() missing 1 required positional argument: 'word_embedding_dimension'